# Citation data

This notebook contains the scripts to get so-called works from [OpenAlex](https://openalex.org/) using [PyAlex](https://github.com/J535D165/pyalex/blob/main/README.md), and scripts for turning the dataset of works into an author-based network. 

The notebook consists of:
1. Methods: loading required packages and Python scripts used to get the works and turn them into an author-based network. The method `query_to_author_network` integrates all the other scripts into a pipeline that takes a query (text and publication year(s)) as input and produces the author-based network. 
2. Scientific episodes: the exact scripts, including the query and publication year filters, used to generate particular author-based networks for the following scientific episodes:
    - Peptic ulcer disease
    - Ego depletion
    - Tobacco: we include two queries at the moment and we'll need to choose one of them

The notebook produces data:
- The resulting data (works and author networks) can be found in this folder and follow the naming convention:
    - `x_works.pkl` contains the works
    - `x_network.pkl` contains the author-based network
- To save and load the data we rely on the `dill` package. The data can be loaded by adapting the following script:
    ```python
    with open("pud_works.pkl", "rb") as f:
        works_pud = dill.load(f)
    ```

⚠️ Important note! [API Key Required](https://developers.openalex.org/guides/authentication) ⚠️
- To utilize the script, you need an OpenAlex API key.
- OpenAlex API  uses a credit-based rate limiting system. 
- The OpenAlex API key is loaded in the ‘Setup’ cell using the `dotenv` package. It requires a `.env` file in the project root;please see `.env.example` for an example. The API key is loaded by the following lines of code:
    ```python
    load_dotenv()  # Loads from .env in project root
    api_key = os.getenv('OPEN_ALEX_API_KEY')
    ```

⚠️ To do ⚠️
1. The code currently contains two queries for the scientific episode concerning tobacco. We must select one. 
2. Archive: The notebook contains a section `Archive`, which contains old code snippets, including also the scientific episode concerning the Perceptron. Should we delete all of this?




## Methods

### Setup

In [23]:
import dill
import copy
import json
import numpy as np
import pandas as pd
import networkx as nx

from collections import Counter
from itertools import chain, chain#, batched
from tqdm.auto import tqdm

from pyalex import Works, Authors, Sources, Institutions, Concepts, Publishers, Funders, config

from IPython.display import display

import os
from dotenv import load_dotenv

load_dotenv()  # Loads from .env in project root
api_key = os.getenv('OPEN_ALEX_API_KEY')

config.max_retries = 5

In [38]:
print(api_key)

nKm2zU6VhTdkc6eDfAhGdp


### Citation data from OpenAlex

In [22]:
def OA_full_text_search(text: str, year: str) -> list[Works]:
    """Searches for works based on the text query and year filtering, using PyAlex. 
    
    Notes: 
    - To search for a multi-word phrase, enclose the phrase in double quotes (e.g., '"ego depletion"'). 
    - To search using Boolean operators, use AND or OR (e.g., "(a OR b) AND (c OR d)").

    Args:
        text (str): The text query to search for.
        year (str): The publication year or range of years to filter by.

    Returns:
        list[Works]: A list of works matching the query and year filter.
    """
    query = Works().search(text).filter(publication_year=year)
    works: list = []

    for _, work in enumerate(chain(*query.paginate(per_page=200, n_max=None))):
        works.append(work)
    return works

### Prune citation data: only articles, and remove articles without bibliography

In [23]:
def get_works_with_references(works: list) -> list:
    works_pruned: list = []
    for work in works:
        try:
            assert work["referenced_works"] != []
            works_pruned.append(work)
        except:
            pass
    return works_pruned

In [24]:
def get_articles(works: list) -> list:
    articles: list = []
    for work in works:
        try:
            assert work["primary_location"]["source"]["type"] == "journal"
            assert work["type"] == "article"
            articles.append(work)
        except:
            pass
    return articles

### Create author network from dataframe of records

In [25]:
def create_author_network(works: list) -> nx.DiGraph:
    """Create a directed author citation network from a list of works.
    Arguments
    ---------
    works : list[Work]
        A list of works (see PyAlex).
    
    Returns
    -------
    G : nx.DiGraph
        A directed graph where nodes are authors and edges represent citations
        from cited authors to citing authors."""
    # Create a directed graph and dataframe of works
    df = pd.DataFrame(works)
    G: nx.DiGraph = nx.DiGraph()

    # Add nodes and edges
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Creating author network"):
        for author in row['authorships']:
            this_author = author['author']['id'] 
            
            # ignore the author with the id "A9999999999" as it is a placeholder for missing values
            if (this_author is None) or (this_author.split("/")[-1] == "A9999999999"): 
                continue
            
            # add node if the author is not already present in the network
            if this_author not in G.nodes():
                G.add_node(this_author)
                G.nodes()[this_author]['n_works'] = 1
            else:
                G.nodes()[this_author]['n_works'] += 1
            
            # Add edges
            for cited_work_id in row["referenced_works"]:
                cited_work = df[df['id'] == cited_work_id] # This fails silently if citations are not present!
                if len(cited_work) >= 1: # In case of multiple hits (shouldn't happen once sampling is fixed)
                    cited_work = cited_work.iloc[0]
                
                for cited_author in cited_work['authorships']:
                    cited_author = cited_author['author']['id'] 
                    
                    if cited_author not in G.nodes():
                        if (cited_author is not None) and (cited_author.split("/")[-1] != "A9999999999"): 
                            G.add_node(cited_author)
                            G.nodes()[cited_author]['n_works'] = 0
                    
                    # edges go FROM cited TO citing
                    if cited_author in G.nodes() and not G.has_edge(cited_author, this_author):
                        G.add_edge(cited_author, this_author)

    return G

### Pruning by removing ‘twins’ (aka, strong co-authors)

In [ ]:
def generate_strong_coauthor_dict(net: nx.DiGraph, records: list) -> dict:
    """Generates a dictionary of strong coauthors for each author in the network. 
    Author B is a strong coauthor of author A if all of A's works have B as a coauthor.

    Args:
        net (nx.DiGraph): The author network.
        records (list): The list of works/articles.

    Returns:
        dict: A dictionary where keys are author IDs and values are sets of strong coauthor IDs.
    """
    strong_coauthors_dict: dict = {}

    for author_id in tqdm(net.nodes(), desc="Generating strong coauthors dict"):
        author_records = [
            work for work in records 
            if author_id in [author["author"]["id"] for author in work["authorships"]]]
        
        strong_coauthors = set()
        for k, record in enumerate(author_records):
            if k == 0:
                coauthors = [
                    coauthor["author"]["id"] 
                    for coauthor in record["authorships"]
                    if coauthor["author"]["id"] != author_id
                ]
                strong_coauthors = set(coauthors)
            elif strong_coauthors == set():
                break
            else:
                coauthors = [
                    coauthor["author"]["id"] 
                    for coauthor in record["authorships"]
                    if coauthor["author"]["id"] != author_id
                ]
                strong_coauthors = strong_coauthors.intersection(set(coauthors))
        if strong_coauthors:
            strong_coauthors_dict[author_id] = strong_coauthors
    return strong_coauthors_dict

In [37]:
def prune_network(
    net: nx.DiGraph, 
    strong_coauthors_dict: dict,
) -> nx.DiGraph:
    """Prunes the author-based network in such a way that the 
    resulting network does not contain any strong co-authors. 

    Args:
        net (nx.DiGraph) 
            The author network.
        strong_coauthors_dict (dict)
            A dictionary where keys are author IDs and values are sets of strong coauthor IDs.

    Returns:
        nx.DiGraph
            The pruned author network.
    """
    network_pruned = copy.deepcopy(net)
    for author_id, strong_coauthors in tqdm(strong_coauthors_dict.items(), desc="Pruning author network"):
        strong_coauthors_in_network = [coauthor for coauthor in strong_coauthors if coauthor in network_pruned.nodes()]
        if strong_coauthors_in_network:
            network_pruned.remove_node(author_id)
    return network_pruned

### Pruning by taking the largest weakly connected component

In [45]:
def produce_lcc(
    net: nx.DiGraph, 
    display_connected_components: bool=True,
) -> nx.DiGraph:
    largest_cc = max(nx.weakly_connected_components(net), key=len)
    
    cc_sizes = [
        len(cc) for cc in sorted(nx.weakly_connected_components(net), key=len, reverse=True)
    ]
    
    if display_connected_components:
        cc_counts = Counter(cc_sizes)
        cc_counts_sorted = sorted(cc_counts.items(), key=lambda x: x[0], reverse=True)
        print(f"Sizes of all weakly connected components:")
        display(pd.DataFrame(cc_counts_sorted, columns=["Size", "Count"]))
    
    lcc = nx.DiGraph()
    lcc.add_nodes_from((n, net.nodes[n]) for n in largest_cc)
    lcc.add_edges_from((n, nbr, d)
        for n, nbrs in net.adj.items() if n in largest_cc
        for nbr, d in nbrs.items() if nbr in largest_cc)
    lcc.graph.update(net.graph)
    return lcc

In [29]:
def remove_self_loops(net: nx.DiGraph) -> nx.DiGraph:
    network_pruned = copy.deepcopy(net).copy()
    for node in net.nodes():
        if (node, node) in net.edges():
            network_pruned.remove_edge(node, node)
    return network_pruned

### Pipeline query -> author network

In [51]:
def query_to_author_network(text, year, filename: str | None = None, show_info: bool = False) -> nx.DiGraph:
    print("Retrieving works from OpenAlex...")
    works = OA_full_text_search(text, year)
    print(f"Retrieved {len(works):,} works from OpenAlex.")
    articles = get_articles(works)
    works_with_refs = get_works_with_references(articles)
    
    author_network_original = create_author_network(articles)
    author_network_pruned = prune_network(
        author_network_original, 
        generate_strong_coauthor_dict(
            author_network_original, 
            articles,
        )
    )
    n_isolates = len(list(nx.isolates(author_network_pruned)))
    print("Extracting largest connected component and removing self-loops...")
    author_network_pruned_lcc = produce_lcc(author_network_pruned)
    author_network_final = remove_self_loops(author_network_pruned_lcc)
        
    if filename is not None:
        with open(f"{filename}_works.pkl", "wb") as f:
            dill.dump(works, f)
        with open(f"{filename}_network.pkl", "wb") as f:
            dill.dump(author_network_final, f)
            
    if show_info: 
        print("-"*50)
        print("Summary of retrieved data and resulting networks")
        print("-"*50)
        info: list[list[str]] = [
            ["works", f"{len(works):,.0f}", "N/A"],
            ["articles", f"{len(articles):,.0f}", "N/A"],
            ["articles with refs", f"{len(works_with_refs):,.0f}", "N/A"],
            [
                "author network", 
                f"{author_network_original.number_of_nodes():,.0f}",
                f"{author_network_original.number_of_edges():,.0f}"
            ],
            [
                "author network pruned", 
                f"{author_network_pruned.number_of_nodes():,.0f}",
                f"{author_network_pruned.number_of_edges():,.0f}"
            ],
            [
                "author network pruned without isolates", 
                f"{author_network_pruned.number_of_nodes() - n_isolates:,}",
                ""
            ],
            [
                "author network pruned lcc", 
                f"{author_network_pruned_lcc.number_of_nodes():,.0f}",
                f"{author_network_pruned_lcc.number_of_edges():,.0f}"
            ],
            [
                "author network final", 
                f"{author_network_final.number_of_nodes():,.0f}",
                f"{author_network_final.number_of_edges():,.0f}"
            ]
        ]
        # info_dict = {
        #     "data_type": 
        #         ["works", 
        #         "articles", 
        #         "articles with refs", 
        #         "author network",
        #         "author network pruned",
        #         "author network pruned without isolates",
        #         "author network pruned lcc",
        #         "author network final"],
        #     "number_of_nodes": [
        #         f"{len(works):,.0f}", 
        #         f"{len(articles):,.0f}", 
        #         f"{len(works_with_refs):,.0f}", 
        #         f"{author_network_original.number_of_nodes():,.0f}", 
        #         f"{author_network_pruned.number_of_nodes():,.0f}", 
        #         f"{author_network_pruned.number_of_nodes() - n_isolates}"
        #         f"{author_network_pruned_lcc.number_of_nodes():,.0f}", 
        #         f"{author_network_final.number_of_nodes():,.0f}"
        #     ],
        #     "number_of_edges": [
        #         "N/A",  # works do not have edges
        #         "N/A",  # articles do not have edges
        #         "N/A",  # articles with refs do not have edges
        #         f"{author_network_original.number_of_edges():,.0f}", 
        #         f"{author_network_pruned.number_of_edges():,.0f}", 
        #         "",
        #         f"{author_network_pruned_lcc.number_of_edges():,.0f}", 
        #         f"{author_network_final.number_of_edges():,.0f}"
        #     ]
        # }
        df_info = pd.DataFrame(info, columns=["data_type", "n_nodes", "n_edges"])
        display(df_info)
    return author_network_final



In [29]:
with open("pud_works.pkl", "rb") as f:
    works = dill.load(f)
with open("pud_network_original.pkl", "rb") as f:
    net = dill.load(f)

In [30]:
articles = get_articles(works)
articles = get_works_with_references(articles)
net = prune_network(net, generate_strong_coauthor_dict(net, articles))

Generating strong coauthors dict:   0%|          | 0/1165 [00:00<?, ?it/s]

Pruning author network:   0%|          | 0/843 [00:00<?, ?it/s]

In [31]:
produce_lcc(net)

Sizes of all weakly connected components:


,Size,Count
0,91,1
1,10,1
2,5,2
3,4,3
4,3,2
5,2,9
6,1,375


# Scientific episodes

### Peptic ulcer disease

For much of the twentieth century, peptic ulcers were attributed to stress, diet, and excess acid secretion — a consensus reinforced by Palmer's influential 1954 study, which concluded that bacteria were not present in the healthy stomach, effectively shutting down inquiry into infectious causes. In 1982, Barry Marshall and Robin Warren identified _Helicobacter pylori_ as the causative agent, but their work was met with widespread skepticism. The medical establishment resisted the finding for over a decade before antibiotic treatment became standard. Marshall and Warren received the Nobel Prize in Physiology or Medicine in 2005.

In [52]:
pud_network = query_to_author_network(
    '"peptic ulcer disease"', 
    "1900-1978", 
    filename="pud", 
    show_info=True,
)

Retrieving works from OpenAlex...
Retrieved 704 works from OpenAlex.


Creating author network:   0%|          | 0/532 [00:00<?, ?it/s]

Generating strong coauthors dict:   0%|          | 0/1165 [00:00<?, ?it/s]

Pruning author network:   0%|          | 0/964 [00:00<?, ?it/s]

Extracting largest connected component and removing self-loops...
Sizes of all weakly connected components:


,Size,Count
0,90,1
1,9,1
2,5,1
3,4,3
4,3,2
5,2,10
6,1,293


--------------------------------------------------
Summary of retrieved data and resulting networks
--------------------------------------------------


,data_type,n_nodes,n_edges
0,works,704,N/A
1,articles,532,N/A
2,articles with refs,411,N/A
3,author network,"1,165","1,513"
4,author network pruned,435,226
5,author network pruned without isolates,145,
6,author network pruned lcc,90,181
7,author network final,90,160


### Ego depletion

Ego depletion refers to the hypothesis, introduced by Roy Baumeister and colleagues in 1998, that self-control draws on a limited mental resource that becomes exhausted with use, i.e., exerting willpower on one task impairs performance on a subsequent task. The idea became one of the most cited findings in social psychology, supported by an early meta-analysis reporting a medium-sized effect. In 2016, a Registered Replication Report coordinated by Hagger and Chatzisarantis across 23 laboratories (N = 2,141) failed to replicate the effect, finding a near-zero effect size.

In [53]:
ego_network = query_to_author_network('"ego depletion"', "1900-2016", filename="ego", show_info=True)

Retrieving works from OpenAlex...
Retrieved 2,054 works from OpenAlex.


Creating author network:   0%|          | 0/948 [00:00<?, ?it/s]

Generating strong coauthors dict:   0%|          | 0/2400 [00:00<?, ?it/s]

Pruning author network:   0%|          | 0/2108 [00:00<?, ?it/s]

Extracting largest connected component and removing self-loops...
Sizes of all weakly connected components:


,Size,Count
0,503,1
1,1,154


--------------------------------------------------
Summary of retrieved data and resulting networks
--------------------------------------------------


,data_type,n_nodes,n_edges
0,works,"2,054",N/A
1,articles,948,N/A
2,articles with refs,824,N/A
3,author network,"2,400","25,405"
4,author network pruned,657,"3,022"
5,author network pruned without isolates,506,
6,author network pruned lcc,503,"3,019"
7,author network final,503,"2,933"


### Tobacco

Epidemiological evidence linking smoking to lung cancer accumulated rapidly in the early 1950s, with landmark studies by Wynder and Graham, Doll and Hill, and Hammond and Horn all showing strong dose-response relationships. The formal institutional consensus came with the 1964 U.S. Surgeon General's Report, _Smoking and Health_, in which an advisory committee reviewed over 7,000 scientific articles and concluded that cigarette smoking causes lung cancer. The decade-long gap between clear evidence and official declaration was partly a product of the tobacco industry's deliberate strategy of manufacturing scientific doubt.

In [54]:
tobacco = query_to_author_network(
    "(tobacco OR smoking OR cigarette) AND (health OR cancer OR lung)", 
    "1900-1964", 
    filename="tobacco", 
    show_info=True,
)

Retrieving works from OpenAlex...
Retrieved 8,214 works from OpenAlex.


Creating author network:   0%|          | 0/3263 [00:00<?, ?it/s]

Generating strong coauthors dict:   0%|          | 0/4060 [00:00<?, ?it/s]

Pruning author network:   0%|          | 0/2300 [00:00<?, ?it/s]

Extracting largest connected component and removing self-loops...
Sizes of all weakly connected components:


,Size,Count
0,289,1
1,9,1
2,5,1
3,4,2
4,3,5
5,2,18
6,1,2018


--------------------------------------------------
Summary of retrieved data and resulting networks
--------------------------------------------------


,data_type,n_nodes,n_edges
0,works,"8,214",N/A
1,articles,"3,263",N/A
2,articles with refs,"1,538",N/A
3,author network,"4,060","4,562"
4,author network pruned,"2,380","1,376"
5,author network pruned without isolates,385,
6,author network pruned lcc,289,"1,292"
7,author network final,289,"1,229"


# Archive

### Peptic ulcer disease

Get the records

In [11]:
# Title and abstract search using OpenAlex 
works_title_2 = OA_title_abstract_search(text="peptic ulcer disease", year="1900-1978") 
print(f"{len(works_title_2)=:,}")
articles_title_2 = get_articles(works_title_2)
print(f"{len(articles_title_2)=:,}")

len(works_title_2)=501
len(articles_title_2)=364


In [ ]:
# Full text search using OpenAlex 
works_pud = OA_full_text_search(text='"peptic ulcer disease"', year="1900-1978")
print(f"{len(works_pud)=:,}") 

len(works_pud)=703


In [ ]:
with open('pud_works.pkl', 'wb') as f:
    dill.dump(works_pud, f)

In [17]:
with open('pud_works.pkl', 'rb') as f:
    works_pud = dill.load(f)

In [ ]:
len(works_pud)

683

In [15]:
works_pud_pruned = get_works_with_references(works_pud)
print(f"{len(works_pud_pruned)=:,}")
articles_pud = get_articles(works_pud_pruned)
print(f"{len(articles_pud)=:,}")

len(works_pud_pruned)=468
len(articles_pud)=409


Create author-based network

In [16]:
network_pud_original = create_author_network(articles_pud) 
print(f"{network_pud_original.number_of_nodes()=:,}")
print(f"{network_pud_original.number_of_edges()=:,}")

network_pud_original.number_of_nodes()=974
network_pud_original.number_of_edges()=1,461


Prune author-based network

In [ ]:
strong_coauthors_dict = generate_strong_coauthor_dict(network_pud_original, works_pud)
network_pud_pruned = prune_network(network_pud_original, strong_coauthors_dict)
print(f"{network_pud_pruned.number_of_nodes()=:,}")
print(f"{network_pud_pruned.number_of_edges()=:,}")

network_pud_pruned_lcc = produce_lcc(network_pud_pruned)
print(f"{network_pud_pruned_lcc.number_of_nodes()=:,}")
print(f"{network_pud_pruned_lcc.number_of_edges()=:,}")

network_pud_final = remove_self_loops(network_pud_pruned_lcc)
print(f"{network_pud_final.number_of_nodes()=:,}")
print(f"{network_pud_final.number_of_edges()=:,}")

  0%|          | 0/974 [00:00<?, ?it/s]

  0%|          | 0/828 [00:00<?, ?it/s]

network_pud_pruned.number_of_nodes()=337
network_pud_pruned.number_of_edges()=224
network_pud_pruned_lcc.number_of_nodes()=87
network_pud_pruned_lcc.number_of_edges()=182
network_pud_final.number_of_nodes()=87
network_pud_final.number_of_edges()=160


,data_type,number_of_nodes,number_of_edges
0,works,695,N/A
1,works with refs,468,N/A
2,articles,409,N/A
3,author network,974,"1,461"
4,author network pruned,337,224
5,author network pruned lcc,87,182
6,author network final,87,160


Save networks

In [20]:
# with open('data/pud_works.pkl', 'w') as f:
#     dill.dump(works_pud, f)

# with open('data/pud_original.pkl', 'wb') as f:
#     dill.dump(network_pud_original, f)

with open('pud_final.pkl', 'wb') as f:
    dill.dump(network_pud_final, f)

# Save as JSON
from networkx.readwrite import json_graph
data_pud = json_graph.node_link_data(network_pud_final)
with open('pud_final.json', 'w') as f:
    json.dump(data_pud, f)

/Users/<user>/Documents/VS Code/GitHub Repositories/e_network_inequality/.venv/lib/python3.10/site-packages/networkx/readwrite/json_graph/node_link.py:142: FutureWarning: 
The default value will be `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_data(G, edges="links") to preserve current behavior, or
  nx.node_link_data(G, edges="edges") for forward compatibility.
  warnings.warn(


Loading the network from file

In [21]:
with open('pud_final.pkl', 'rb') as f:
    network = dill.load(f)

### Ego depletion theory

In [34]:
works_ego = OA_full_text_search(text="ego depletion", year="1900-2016")
print(f"{len(works_ego)=:,}")


len(works_ego)=2,047


In [13]:
works_ego_pruned = get_works_with_references(works_ego)
print(f"{len(works_ego_pruned)=:,}")
works_ego_articles = get_articles(works_ego_pruned)
print(f"{len(works_ego_articles)=:,}")

len(works_ego_pruned)=1,304
len(works_ego_articles)=794


In [ ]:
network_ego_original = create_author_network(works_ego_articles)
print(f"{network_ego_original.number_of_nodes()=:,}")
print(f"{network_ego_original.number_of_edges()=:,}")

network_ego_pruned = prune_network(network_ego_original, generate_strong_coauthor_dict(network_ego_original, works_ego))
print(f"{network_ego_pruned.number_of_nodes()=:,}")
network_ego_pruned_lcc = produce_lcc(network_ego_pruned)
print(f"{network_ego_pruned_lcc.number_of_nodes()=:,}")
network_ego_final = remove_self_loops(network_ego_pruned_lcc)
print(f"{network_ego_final.number_of_nodes()=:,}")

network_ego_original.number_of_nodes()=2,040
network_ego_original.number_of_edges()=24,770


  0%|          | 0/2040 [00:00<?, ?it/s]

  0%|          | 0/1731 [00:00<?, ?it/s]

network_ego_pruned.number_of_nodes()=597
network_ego_pruned_lcc.number_of_nodes()=515
network_ego_final.number_of_nodes()=515


In [15]:
print(f"{network_ego_final.number_of_edges()=:,}")

network_ego_final.number_of_edges()=3,444


### Alternative PUD network

In [22]:
with open('pud_works.pkl', 'rb') as f:
    works_pud = dill.load(f)

In [23]:
works_pud_pruned = get_works_with_references(works_pud)
print(f"{len(works_pud_pruned)=:,}")
articles_pud = get_articles(works_pud_pruned)
print(f"{len(articles_pud)=:,}")

len(works_pud_pruned)=464
len(articles_pud)=407


In [24]:
network_pud_original = create_author_network(articles_pud) 
print(f"{network_pud_original.number_of_nodes()=:,}")
print(f"{network_pud_original.number_of_edges()=:,}")

network_pud_original.number_of_nodes()=974
network_pud_original.number_of_edges()=1,461


In [ ]:
strong_coauthors_dict = generate_strong_coauthor_dict(network_pud_original, works_pud)
network_pud_pruned = prune_network(network_pud_original, strong_coauthors_dict)
print(f"{network_pud_pruned.number_of_nodes()=:,}")
print(f"{network_pud_pruned.number_of_edges()=:,}")

network_pud_no_loops = remove_self_loops(network_pud_pruned)
print(f"{network_pud_no_loops.number_of_nodes()=:,}")
print(f"{network_pud_no_loops.number_of_edges()=:,}")

100%|██████████| 831/831 [00:00<00:00, 255233.35it/s]

network_pud_pruned.number_of_nodes()=337
network_pud_pruned.number_of_edges()=224
network_pud_no_loops.number_of_nodes()=337
network_pud_no_loops.number_of_edges()=196


In [26]:
import copy

def remove_zero_indegree_nodes(network: nx.DiGraph) -> nx.DiGraph:
    net_result = copy.deepcopy(network)
    for k in range(10_000):
        nodes_to_remove = [
            node for node in net_result.nodes() 
            if net_result.in_degree(node) == 0
        ]
        net_result.remove_nodes_from(nodes_to_remove)
        if nodes_to_remove == []:
            break
    return net_result

In [27]:
network_pud_nonzero_indegree = remove_zero_indegree_nodes(network_pud_no_loops)
print(f"{network_pud_nonzero_indegree.number_of_nodes()=:,}")
print(f"{network_pud_nonzero_indegree.number_of_edges()=:,}")

network_pud_nonzero_indegree.number_of_nodes()=32
network_pud_nonzero_indegree.number_of_edges()=65


In [28]:
network_lcc = produce_lcc(network_pud_nonzero_indegree)
print(f"{network_lcc.number_of_nodes()=:,}")
print(f"{network_lcc.number_of_edges()=:,}")

network_lcc.number_of_nodes()=32
network_lcc.number_of_edges()=65


In [29]:
with open('pud_alternative.pkl', 'wb') as f:
    dill.dump(network_lcc, f)

### Perceptron

In [30]:
string = "perceptron"
works_perceptron = OA_full_text_search(text=string, year="1900-1971") 

In [31]:
works_perceptron_pruned = get_works_with_references(works_perceptron)
print(f"{len(works_perceptron_pruned)=:,}")

len(works_perceptron_pruned)=101


In [32]:
articles_perceptron = get_articles(works_perceptron_pruned)
print(f"{len(articles_perceptron)=:,}")

len(articles_perceptron)=74


Create author-based network

In [33]:
network_perceptron_original = create_author_network(articles_perceptron) 
print(f"{network_perceptron_original.number_of_nodes()=:,}")
print(f"{network_perceptron_original.number_of_edges()=:,}")

network_perceptron_original.number_of_nodes()=116
network_perceptron_original.number_of_edges()=49


Prune author-based network

In [ ]:
strong_coauthors_dict = generate_strong_coauthor_dict(network_perceptron_original, articles_perceptron)
network_perceptron_pruned = prune_network(network_perceptron_original, strong_coauthors_dict)
print(f"{network_perceptron_pruned.number_of_nodes()=:,}")
print(f"{network_perceptron_pruned.number_of_edges()=:,}")

network_perceptron_pruned_lcc = produce_lcc(network_perceptron_pruned)
print(f"{network_perceptron_pruned_lcc.number_of_nodes()=:,}")
print(f"{network_perceptron_pruned_lcc.number_of_edges()=:,}")

network_perceptron_final = remove_self_loops(network_perceptron_pruned_lcc)
print(f"{network_perceptron_final.number_of_nodes()=:,}")
print(f"{network_perceptron_final.number_of_edges()=:,}")

100%|██████████| 79/79 [00:00<00:00, 181164.58it/s]

network_perceptron_pruned.number_of_nodes()=64
network_perceptron_pruned.number_of_edges()=19
network_perceptron_pruned_lcc.number_of_nodes()=10
network_perceptron_pruned_lcc.number_of_edges()=12
network_perceptron_final.number_of_nodes()=10
network_perceptron_final.number_of_edges()=9


Save networks

In [35]:
# with open('data/perceptron_works.pkl', 'wb') as f:
#     dill.dump(works_perceptron, f)

# with open('data/perceptron_original.pkl', 'wb') as f:
#     dill.dump(network_perceptron_original, f)
    
with open('perceptron_final_dill.pkl', 'wb') as f:
    dill.dump(network_perceptron_final, f)

# Save the object to a file
with open('perceptron_final.pkl', 'wb') as f:
    pickle.dump(network_perceptron_final, f)

# Save as JSON
data_perceptron = json_graph.node_link_data(network_perceptron_final)
with open('perceptron_final.json', 'w') as f:
    json.dump(data_perceptron, f)

In [36]:
info_dict = {
    "data_type": 
        ["works", 
        "works with refs", 
        "articles", 
        "author network",
        "author network pruned",
        "author network pruned lcc",
        "author network final"],
    "number_of_nodes": [
        f"{len(works_perceptron):,.0f}", 
        f"{len(works_perceptron_pruned):,.0f}", 
        f"{len(articles_perceptron):,.0f}", 
        f"{network_perceptron_original.number_of_nodes():,.0f}", 
        f"{network_perceptron_pruned.number_of_nodes():,.0f}", 
        f"{network_perceptron_pruned_lcc.number_of_nodes():,.0f}", 
        f"{network_perceptron_final.number_of_nodes():,.0f}"
    ],
    "number_of_edges": [
        "N/A",  # works do not have edges
        "N/A",  # works with refs do not have edges
        "N/A",  # articles do not have edges
        f"{network_perceptron_original.number_of_edges():,.0f}", 
        f"{network_perceptron_pruned.number_of_edges():,.0f}", 
        f"{network_perceptron_pruned_lcc.number_of_edges():,.0f}", 
        f"{network_perceptron_final.number_of_edges():,.0f}"
    ]
}
df_info = pd.DataFrame(info_dict)
# df_info.astype({"number_of_edges": "Int64"})
df_info

,data_type,number_of_nodes,number_of_edges
0,works,195,N/A
1,works with refs,101,N/A
2,articles,74,N/A
3,author network,116,49
4,author network pruned,64,19
5,author network pruned lcc,10,12
6,author network final,10,9


### Perceptron

In [37]:
# Use the new API function OA_full_text_search (version 1 to match old behavior or default logic)
works_perceptron = OA_full_text_search(text="perceptron", year="1900-2000", version=1)
print(f"{len(works_perceptron)=:,}")

len(works_perceptron)=6,212


In [38]:
works_perceptron_pruned = get_works_with_references(works_perceptron)
print(f"{len(works_perceptron_pruned)=:,}")

len(works_perceptron_pruned)=3,933


In [39]:
articles_perceptron = get_articles(works_perceptron_pruned)
print(f"{len(articles_perceptron)=:,}")

len(articles_perceptron)=2,386


Create author-based network

In [40]:
network_perceptron_original = create_author_network(articles_perceptron) 
print(f"{network_perceptron_original.number_of_nodes()=:,}")
print(f"{network_perceptron_original.number_of_edges()=:,}")

network_perceptron_original.number_of_nodes()=4,263
network_perceptron_original.number_of_edges()=12,665


Prune author-based network

In [ ]:
# Note: The old notebook wrapped this in tqdm(..., items()) inside the function,
# and the new notebook also has tqdm inside the function.
# We pass 'articles_perceptron' as the records list.
strong_coauthors_dict = generate_strong_coauthor_dict(network_perceptron_original, articles_perceptron)
network_perceptron_pruned = prune_network(network_perceptron_original, strong_coauthors_dict)
print(f"{network_perceptron_pruned.number_of_nodes()=:,}")
print(f"{network_perceptron_pruned.number_of_edges()=:,}")

network_perceptron_pruned_lcc = produce_lcc(network_perceptron_pruned)
print(f"{network_perceptron_pruned_lcc.number_of_nodes()=:,}")
print(f"{network_perceptron_pruned_lcc.number_of_edges()=:,}")

network_perceptron_final = remove_self_loops(network_perceptron_pruned_lcc)
print(f"{network_perceptron_final.number_of_nodes()=:,}")
print(f"{network_perceptron_final.number_of_edges()=:,}")

100%|██████████| 3527/3527 [00:00<00:00, 95476.44it/s]


network_perceptron_pruned.number_of_nodes()=1,507
network_perceptron_pruned.number_of_edges()=3,320
network_perceptron_pruned_lcc.number_of_nodes()=801
network_perceptron_pruned_lcc.number_of_edges()=3,224
network_perceptron_final.number_of_nodes()=801
network_perceptron_final.number_of_edges()=2,983


Save networks

In [42]:
# with open('data/perceptron_works.pkl', 'wb') as f:
#     dill.dump(works_perceptron, f)

# with open('data/perceptron_original.pkl', 'wb') as f:
#     dill.dump(network_perceptron_original, f)
    
with open('perceptron_final_dill.pkl', 'wb') as f:
    dill.dump(network_perceptron_final, f)

# Save the object to a file
with open('perceptron_final.pkl', 'wb') as f:
    pickle.dump(network_perceptron_final, f)

# Save as JSON
data_perceptron = json_graph.node_link_data(network_perceptron_final)
with open('perceptron_final.json', 'w') as f:
    json.dump(data_perceptron, f)

/Users/<user>/Documents/VS Code/GitHub Repositories/e_network_inequality/.venv/lib/python3.10/site-packages/networkx/readwrite/json_graph/node_link.py:142: FutureWarning: 
The default value will be `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_data(G, edges="links") to preserve current behavior, or
  nx.node_link_data(G, edges="edges") for forward compatibility.
  warnings.warn(
